# Detailed EDA - Hate Speech Dataset

This notebook performs a detailed exploratory data analysis (EDA) on the raw and cleaned tweet dataset.

## 1) Imports and Settings

In [ ]:
from pathlib import Path
from collections import Counter
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 200)
RANDOM_STATE = 42

## 2) Load Data

In [ ]:
RAW_PATH = Path("data/raw/labeled_data.csv")
CLEANED_PATH = Path("data/processed/cleaned_labeled_data.csv")

if not RAW_PATH.exists():
    raise FileNotFoundError(f"Raw dataset not found at {RAW_PATH}")

df_raw = pd.read_csv(RAW_PATH)

if CLEANED_PATH.exists():
    df_cleaned = pd.read_csv(CLEANED_PATH)
else:
    df_cleaned = None

print("Raw shape:", df_raw.shape)
print("Cleaned available:", df_cleaned is not None)
if df_cleaned is not None:
    print("Cleaned shape:", df_cleaned.shape)

## 3) Schema and Basic Quality Checks

In [ ]:
display(df_raw.head(5))
print("\nColumns:", list(df_raw.columns))
print("\nDtypes:\n")
print(df_raw.dtypes)

print("\nMissing values:\n")
display(df_raw.isna().sum().to_frame("missing_count"))

duplicate_tweets = df_raw["tweet"].duplicated().sum() if "tweet" in df_raw.columns else 0
print(f"Duplicate tweet rows (by text): {duplicate_tweets}")

## 4) Class Distribution and Imbalance

In [ ]:
label_map = {0: "hate_speech", 1: "offensive_language", 2: "neither"}

if "class" not in df_raw.columns:
    raise KeyError("Expected column 'class' in raw dataset")

class_counts = df_raw["class"].value_counts().sort_index()
class_df = pd.DataFrame({
    "class": class_counts.index,
    "label": class_counts.index.map(lambda x: label_map.get(x, f"class_{x}")),
    "count": class_counts.values,
    "percent": (class_counts.values / len(df_raw) * 100).round(2),
})
display(class_df)

minority_ratio = class_counts.min() / class_counts.max()
print(f"Minority/Majority ratio: {minority_ratio:.4f}")

plt.figure(figsize=(8, 4))
ax = sns.barplot(data=class_df, x="label", y="count", palette="viridis")
ax.set_title("Class Distribution")
ax.set_xlabel("Class")
ax.set_ylabel("Count")
for i, v in enumerate(class_df["count"]):
    ax.text(i, v, str(v), ha="center", va="bottom", fontsize=9)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 5) Tweet Length Analysis

In [ ]:
df_len = df_raw.copy()
df_len["tweet"] = df_len["tweet"].astype(str)
df_len["char_len"] = df_len["tweet"].str.len()
df_len["word_len"] = df_len["tweet"].str.split().str.len()

display(df_len[["char_len", "word_len"]].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99]))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df_len["char_len"], bins=50, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Distribution of Tweet Character Length")
axes[0].set_xlabel("Characters")

sns.histplot(df_len["word_len"], bins=40, kde=True, ax=axes[1], color="darkorange")
axes[1].set_title("Distribution of Tweet Word Length")
axes[1].set_xlabel("Words")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=df_len, x="class", y="char_len", ax=axes[0], palette="Set2")
axes[0].set_title("Character Length by Class")
axes[0].set_xlabel("Class")

sns.boxplot(data=df_len, x="class", y="word_len", ax=axes[1], palette="Set3")
axes[1].set_title("Word Length by Class")
axes[1].set_xlabel("Class")
plt.tight_layout()
plt.show()

## 6) Pattern Counts in Raw Tweets

In [ ]:
URL_RE = re.compile(r"https?://\\S+|www\\.\\S+", re.IGNORECASE)
MENTION_RE = re.compile(r"@\\w+")
HASHTAG_RE = re.compile(r"#\\w+")
RT_RE = re.compile(r"\\bRT\\b", re.IGNORECASE)

df_pat = df_raw.copy()
df_pat["tweet"] = df_pat["tweet"].astype(str)
df_pat["has_url"] = df_pat["tweet"].str.contains(URL_RE)
df_pat["has_mention"] = df_pat["tweet"].str.contains(MENTION_RE)
df_pat["has_hashtag"] = df_pat["tweet"].str.contains(HASHTAG_RE)
df_pat["has_rt"] = df_pat["tweet"].str.contains(RT_RE)

pattern_summary = pd.DataFrame({
    "feature": ["has_url", "has_mention", "has_hashtag", "has_rt"],
    "count": [
        int(df_pat["has_url"].sum()),
        int(df_pat["has_mention"].sum()),
        int(df_pat["has_hashtag"].sum()),
        int(df_pat["has_rt"].sum()),
    ],
})
pattern_summary["percent"] = (pattern_summary["count"] / len(df_pat) * 100).round(2)
display(pattern_summary)

plt.figure(figsize=(8, 4))
ax = sns.barplot(data=pattern_summary, x="feature", y="percent", palette="magma")
ax.set_title("Tweet Patterns in Raw Data")
ax.set_ylabel("Percent of tweets")
ax.set_xlabel("")
plt.tight_layout()
plt.show()

## 7) Frequent Tokens and N-grams (Raw vs Cleaned)

In [ ]:
def top_ngrams(corpus, ngram_range=(1, 1), top_k=20, min_df=2):
    vec = CountVectorizer(stop_words="english", ngram_range=ngram_range, min_df=min_df)
    x = vec.fit_transform(corpus)
    counts = np.asarray(x.sum(axis=0)).ravel()
    terms = np.array(vec.get_feature_names_out())
    idx = counts.argsort()[::-1][:top_k]
    return pd.DataFrame({"term": terms[idx], "count": counts[idx]})

raw_text = df_raw["tweet"].astype(str)
raw_unigrams = top_ngrams(raw_text, ngram_range=(1, 1), top_k=20)
raw_bigrams = top_ngrams(raw_text, ngram_range=(2, 2), top_k=20)

display(raw_unigrams)
display(raw_bigrams.head(10))

In [ ]:
if df_cleaned is not None and "clean_tweet" in df_cleaned.columns:
    clean_text = df_cleaned["clean_tweet"].fillna("").astype(str)
    clean_unigrams = top_ngrams(clean_text, ngram_range=(1, 1), top_k=20)
    clean_bigrams = top_ngrams(clean_text, ngram_range=(2, 2), top_k=20)
    display(clean_unigrams)
    display(clean_bigrams.head(10))
else:
    print("Cleaned dataset/column not available. Run preprocess_data.py first.")

## 8) Class-wise Top Terms (Cleaned Text Preferred)

In [ ]:
source_df = df_cleaned if (df_cleaned is not None and "clean_tweet" in df_cleaned.columns) else df_raw.copy()
text_col = "clean_tweet" if "clean_tweet" in source_df.columns else "tweet"
source_df[text_col] = source_df[text_col].fillna("").astype(str)

for cls in sorted(source_df["class"].unique()):
    subset = source_df[source_df["class"] == cls][text_col]
    top_terms = top_ngrams(subset, ngram_range=(1, 1), top_k=15, min_df=2)
    print(f"\nClass {cls} ({label_map.get(cls, f'class_{cls}')}) - top terms")
    display(top_terms)

## 9) Short Error-Prone or Extreme Samples

In [ ]:
tmp = df_raw.copy()
tmp["tweet"] = tmp["tweet"].astype(str)
tmp["word_len"] = tmp["tweet"].str.split().str.len()

print("Very short tweets (<= 3 words):")
display(tmp[tmp["word_len"] <= 3][["class", "tweet"]].head(15))

print("Very long tweets (top 1% by word length):")
threshold = tmp["word_len"].quantile(0.99)
display(tmp[tmp["word_len"] >= threshold][["class", "tweet", "word_len"]].head(15))

## 10) Summary Notes Template

### Filled Summary (Current Dataset)

- Class imbalance severity: Strong imbalance is present. Class counts are `0: 1430` (5.77%), `1: 19190` (77.43%), `2: 4163` (16.80%). Minority/majority ratio is ~`0.0745`.
- Evidence of noisy patterns: Raw tweets contain high social-media noise: mentions in `57.25%` of rows, hashtags in `30.80%`, retweet token `RT` in `28.89%`, URLs in `12.04%`.
- Length characteristics: Average tweet length is about `14.12` words; 99th percentile is around `29` words. The dataset includes very short texts and long-tail tweets that may be harder to classify consistently.
- Most informative terms/n-grams per class: Class-wise frequent-token sections show overlap between class `0` and class `1`, which explains confusion between hate speech and offensive language in model results.
- Suggested modeling actions:
  1. Keep class-aware training (`class_weight='balanced'` or custom weights).
  2. Prioritize `f1_macro` and class-0 metrics (not accuracy alone).
  3. Use threshold tuning / decision calibration for class `0` to improve hate-speech recall-precision tradeoff.
  4. Add richer text features (word + char n-grams) and compare with tuned LinearSVC.
  5. Continue cleaning social artifacts and inspect frequent false positives/false negatives for label-boundary ambiguity.